In [ ]:
## Exercice 3 — Modéliser CQRS pour un cas métier

# ? Une bibliothèque numérique.
# ? Les adhérents empruntent des livres (e-books), notent, ajoutent en favoris.
# ? Les bibliothécaires ajoutent des ouvrages au catalogue, gèrent les retours, suspendent un compte en cas d'abus.
# ? Les administrateurs consultent des statistiques d'usage, font des rapports trimestriels au conseil municipal.
# * 3 commands clairement nommées, avec leurs champs (côté écriture).
# * 3 vues (queries) distinctes, une par persona (adhérent / bibliothécaire / administrateur). Préciser pour chacune les champs exposés.
# * Une règle métier d'écriture côté commands qui ne doit pas apparaître côté reads.
# * Une règle de sécurité qui filtre ce qu'un adhérent peut voir, exprimée comme une différence de vue plutôt qu'un contrôle d'accès.

# ! séparateur ! #
# ! séparateur ! #
# ! séparateur ! #

# * adhérents - commandes : créer emprunt, noter, ajouter en favoris
# * bibliothécaires - commandes : ajouter ouvrage, créer retour, suspendre compte ; requêtes : consulter retours
# * administrateurs - commandes : faire rapports trimestriels ; requêtes : statistiques d'usage

from typing import Any


# ! Commandes et requêtes pour les adhérents

class CreateBorrowCommand:
    # ! règle métier : un adhérent ne peut pas emprunter plus de 5 livres simultanément
    user_id: int
    book_id: int # * à écrire

class ReadUserBorrowedBooksQuery:
    user_id: int
    # ? expose la liste des livres empruntés par l'utilisateur, avec titre, auteur, date d'emprunt et date de retour prévue

class RateBookCommand:
    user_id: int
    book_id: int
    rating: int # * à écrire

class ReadBookRatingsQuery:
    user_id: int
    # ? expose la liste des livres notés par l'utilisateur, avec titre, auteur, note attribuée et date de notation

class AddToFavoritesCommand:
    user_id: int
    book_id: int # * à écrire

class ReadFavoriteBooksQuery:
    user_id: int
    # ? expose la liste des livres ajoutés en favoris par l'utilisateur, avec titre, auteur et date d'ajout


# ! Commandes et requêtes pour les bibliothécaires

class AddBookCommand:
    librarian_id: int
    book_details: dict[str, Any] # * à écrire

class ReadCatalogQuery:
    librarian_id: int
    # ? expose la liste des livres du catalogue, avec titre, auteur, date d'ajout et disponibilité

class ReadAllBorrowedBooksQuery:
    librarian_id: int
    # ? expose la liste des livres empruntés, avec titre, nom de l'adhérent, date d'emprunt et date de retour prévue

class ManageReturnCommand:
    librarian_id: int
    user_id: int
    book_id: int # * à écrire

class ReadOverdueReturnsQuery:
    librarian_id: int
    # ? expose la liste des retours en retard, avec titre du livre, nom de l'adhérent, date d'emprunt et date de retour prévue

class SuspendAccountCommand:
    librarian_id: int
    user_id: int # * à écrire

class UnsuspendAccountCommand:
    librarian_id: int
    user_id: int # * à écrire

class ReadSuspendedAccountsQuery:
    librarian_id: int
    # ? expose la liste des comptes suspendus, avec nom de l'adhérent, date de suspension et raison

class ReadReturnsQuery:
    librarian_id: int
    # ? expose la liste des retours attendus, avec titre du livre, nom de l'adhérent, date d'emprunt et date de retour prévue


# ! Commandes et requêtes pour les administrateurs

class GenerateQuarterlyReportCommand:
    admin_id: int

class ReadQuarterlyReportQuery:
    admin_id: int
    # ? expose le rapport trimestriel généré, avec statistiques d'emprunts, retours, notes et favoris

class ReadGeneralStatisticsQuery:
    admin_id: int
    # ? expose les statistiques d'usage, avec nombre total d'emprunts, livres populaires, etc.

In [ ]:
from typing import Optional

class ConsumeCreditCommand:
   user_id: int
   amount: float

class ConsumeCreditResult:
   ok: bool
   reason: Optional[str] = None

class ReadUserCreditQuery:
   user_id: int

class ReadUserCreditResult:
   credit: float

class ConsumeCreditHandler:
   def __init__(self, db: Any):
      self.db = db

   def handle_command(self, command: ConsumeCreditCommand) -> ConsumeCreditResult:
      user = self.db.users.find_one({"_id": command.user_id})
      if user["credit"] < command.amount:
         return ConsumeCreditResult(ok=False, reason="Insufficient credit")
      self.db.users.update_one(
         {"_id": command.user_id},
         {"$inc": {"credit": -command.amount}},
      )
      return ConsumeCreditResult(ok=True)

class ReadUserCreditHandler:
   def __init__(self, db: Any):
      self.db = db

   def handle_query(self, query: ReadUserCreditQuery) -> float:
      user = self.db.users.find_one({"_id": query.user_id})
      return user["credit"]

db = None  # Placeholder for the database connection

def consume_credit(user_id: int, amount: float) -> ConsumeCreditResult:
   handler = ConsumeCreditHandler(db)
   command = ConsumeCreditCommand(user_id=user_id, amount=amount)
   result = handler.handle_command(command)
   return result

def get_user_credit(user_id: int) -> ReadUserCreditResult:
   handler = ReadUserCreditHandler(db)
   query = ReadUserCreditQuery(user_id=user_id)
   result = handler.handle_query(query)
   return ReadUserCreditResult(credit=result)